#Import Packagese

In [16]:
from collections import defaultdict
import numpy as np
import torch
import torchvision
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import os
import copy
from torch.utils.data import random_split
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


#Load Dataset

In [15]:
def load_Dataset(target_classes, train_transform, val_transform, root='./cifar_100'):
    cifar100_path = os.path.join(root, 'cifar-100-python')

    if os.path.isdir(cifar100_path):
        print(f"Dataset found in '{root}'. Loading from local files.")
    else:
        print(f"Dataset not found in '{root}'. Downloading...")
    train_dataset_full = torchvision.datasets.CIFAR100(root=root,transform=train_transform,train=True,download=True)
    test_dataset_full = torchvision.datasets.CIFAR100(root=root,transform=val_transform,train=False,download=True)

    print("Dataset Downloaded Successfully ")
    all_classes = train_dataset_full.classes
    try:
        target_indices = [all_classes.index(cls) for cls in target_classes]
    except ValueError as e:
        print(f"Error : {e}")
        return None,None
    label_map = {old_label: new_label for new_label, old_label in enumerate(target_indices)}
    def _filter_dataset(dataset):

        targets_np = np.array(dataset.targets)

        indices_to_keep = np.isin(targets_np, target_indices)


        dataset.data = dataset.data[indices_to_keep]


        original_targets_to_keep = targets_np[indices_to_keep]

        dataset.targets = [label_map[target] for target in original_targets_to_keep]


        dataset.classes = target_classes
        return dataset
    print(f"\nFiltering for {len(target_classes)} classes...")

    train_dataset_subset = _filter_dataset(train_dataset_full)

    test_dataset_subset = _filter_dataset(test_dataset_full)
    print("Filtering complete. Returning training and validation datasets.")


    return train_dataset_subset, test_dataset_subset


In [3]:
mean = (0.5071, 0.4867, 0.4408)
std = (0.2675, 0.2565, 0.2761)
val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,std=std)
])
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean,std=std)
])

In [4]:
subset_target_classes = [
    # Flowers
    'orchid', 'poppy', 'sunflower',
    # Mammals
    'fox', 'raccoon', 'skunk',
    # Insects
    'butterfly', 'caterpillar', 'cockroach'
]

In [17]:
train_dataset_proto, val_dataset_proto = load_Dataset(subset_target_classes, train_transform, val_transform)

Dataset found in './cifar_100'. Loading from local files.
Dataset Downloaded Successfully 

Filtering for 9 classes...
Filtering complete. Returning training and validation datasets.


In [18]:

batch_size = 64
val_size = int(0.5 * len(val_dataset_proto))
test_size = len(val_dataset_proto) - val_size
val_split_proto , test_split_proto = random_split(val_dataset_proto, [val_size,test_size])
train_loader = DataLoader(train_dataset_proto, batch_size=batch_size, shuffle=True)

val_loader = DataLoader(val_split_proto, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_split_proto,batch_size=100,shuffle=False)

#Model

In [19]:
class SimpleCNN(nn.Module):

    def __init__(self, num_classes):


        super(SimpleCNN, self).__init__()


        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)


        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)


        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)


        self.flatten = nn.Flatten()



        self.fc1 = nn.Linear(128 * 4 * 4, 512)
        self.relu4 = nn.ReLU()
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, num_classes)


    def forward(self, x):


        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)


        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)


        x = self.conv3(x)
        x = self.relu3(x)
        x = self.pool3(x)


        x = self.flatten(x)


        x = self.fc1(x)
        x = self.relu4(x)
        x = self.dropout(x)
        x = self.fc2(x)


        return x

In [20]:
# Get the number of classes
num_classes = len(train_dataset_proto.classes)

# Instantiate the model
model = SimpleCNN(num_classes)

In [21]:
# Loss function
loss_function = nn.CrossEntropyLoss()

# Optimizer for the prototype model
optimizer = optim.Adam(model.parameters(), lr=0.001)

#Train and Evaluate

In [22]:
model.to(device)
def train_model(model, train_loader, loss_function, optimizer,device):

  model.train()

  for image,label in train_loader:
    image,label = image.to(device),label.to(device)
    optimizer.zero_grad()
    output = model(image)
    loss = loss_function(output,label)
    loss.backward()
    optimizer.step()


In [23]:
def evaluate_model(model, val_loader,device):
  model.eval()
  correct = 0
  total = 0
  with torch.no_grad():
    for image,label in val_loader:
      image,label = image.to(device),label.to(device)
      output = model(image)
      _,preds = output.max(1)
      total += label.size(0)
      correct += preds.eq(label).sum().item()
  return 100. * correct / total

In [24]:
epochs = 20
best_model = None
best_accuracy = 0.0
best_epoch = 0
for epoch in range(epochs):
  print(f"Epoch : {epoch+1}")
  train_model(model,train_loader,loss_function,optimizer,device)
  accuracy = evaluate_model(model,val_loader,device)
  if accuracy > best_accuracy:
    best_epoch = epoch+1
    best_accuracy = accuracy
    best_model = copy.deepcopy(model)
  print(f"Test Accuracy :{accuracy:.2f}")
print(f"The Best Model Accuracy is {best_accuracy:.2f} at Epoch {best_epoch}")

Epoch : 1
Test Accuracy :53.56
Epoch : 2
Test Accuracy :64.67
Epoch : 3
Test Accuracy :69.56
Epoch : 4
Test Accuracy :70.44
Epoch : 5
Test Accuracy :74.22
Epoch : 6
Test Accuracy :74.44
Epoch : 7
Test Accuracy :76.00
Epoch : 8
Test Accuracy :78.00
Epoch : 9
Test Accuracy :77.33
Epoch : 10
Test Accuracy :76.89
Epoch : 11
Test Accuracy :76.67
Epoch : 12
Test Accuracy :78.00
Epoch : 13
Test Accuracy :78.22
Epoch : 14
Test Accuracy :78.44
Epoch : 15
Test Accuracy :78.89
Epoch : 16
Test Accuracy :79.11
Epoch : 17
Test Accuracy :78.89
Epoch : 18
Test Accuracy :79.33
Epoch : 19
Test Accuracy :80.00
Epoch : 20
Test Accuracy :79.78
The Best Model Accuracy is 80.00 at Epoch 19


#Testing

In [25]:
best_model.eval()
correct = 0
total = 0
accuracy = 0.0
with torch.no_grad():
  for image,label in test_loader:
    image,label = image.to(device),label.to(device)
    outputs = best_model(image)
    _,preds = outputs.max(1)
    correct += (preds==label).sum().item()
    total += label.size(0)
  accuracy = 100. * correct / total
print(f"The Model Accuracy on Test Dataset is: {accuracy:.2f}")

The Model Accuracy on Test Dataset is: 78.22
